# Semantic retrieval with sentence embeddings

This notebook uses sentence embeddings to rank parliamentary sentences by their similarity to a research query.
The substantive task is to find statements about **housing affordability** in the ParlEE UK CAP data.

By the end, you should be able to

- embed a query and rank a corpus by cosine similarity;
- define relevance for a research question rather than treating similarity as a
  ready-made measurement;
- assess how stable the ranking is when the same concept is phrased differently;
- distinguish precision among highly ranked results from recall and prevalence.

<br><a target="_blank" href="https://colab.research.google.com/github/haukelicht/advanced_text_analysis/blob/main/notebooks/embedding/retrieval.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

In [ ]:
COLAB = True
try:
    import google.colab
except ImportError:
    COLAB = False

if COLAB:
    !git clone --branch main --single-branch --depth 1 --filter=blob:none https://github.com/haukelicht/advanced_text_analysis.git
    !pip install -q sentence-transformers~=6.1.0 seaborn~=0.13.2

In [ ]:
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

In [ ]:
MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"

base_path = Path("/content/advanced_text_analysis" if COLAB else "../..")

data_path = base_path / "data/labeled/sylvester_parlee_2022"
data_file = "sylvester_parlee_2022-uk_cap_sentences.csv"

results_path = base_path / "notebooks/embedding/cache"

## Load the corpus

The prepared file contains one row per unique sentence and a stable `text_id`. The
CAP topic code is retained for a later benchmark, but it plays no role in ranking
the sentences.

In [ ]:
if data_file.exists():
    df = pd.read_csv(data_file)
else:
    # The cached corpus provides a fallback when the prepared CSV is unavailable.
    with open(results_path / "retrieval_corpus.pkl", "rb") as file:
        df = pickle.load(file)

df = df.dropna(subset=["text_id", "text", "cap_topic"]).reset_index(drop=True)
if not df["text_id"].is_unique:
    raise ValueError("Expected one row per stable text_id.")

df[["text_id", "text"]].head()

## Embed the sentences

We use the same compact English sentence-embedding model as in the preceding
exercise. Its maximum sequence length is a token limit: longer inputs are truncated
before the model produces an embedding.

In [ ]:
# Most of the required code is provided. Run it and inspect the model limit.
embedding_model = SentenceTransformer(MODEL_ID)
embedding_model.max_seq_length

In [ ]:
token_lengths = embedding_model.tokenizer(
    df["text"].tolist(),
    return_length=True,
    truncation=False,
)["length"]

pd.Series({
    "longest input": max(token_lengths),
    "model limit": embedding_model.max_seq_length,
    "share truncated": np.mean(
        np.asarray(token_lengths) > embedding_model.max_seq_length
    ),
})

In [ ]:
# TODO: Run this cell to embed every sentence. Depending on the computer, this may
# take a few minutes.
embeddings = embedding_model.encode(
    df["text"].tolist(),
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True,
)
embeddings.shape

Normalizing the embeddings to unit length means that their matrix product equals
cosine similarity. Row `i` of `embeddings` remains aligned with row `i` of `df`.

::: {.callout-tip title="Fallback: load cached embeddings or rankings"}

Try the embedding and ranking code yourself first. If model download or computation
fails, you can use either fallback:

1. **Continue the calculations with cached embeddings.** This file contains the
   corpus embeddings and the two query embeddings. It also records the corpus row
   order so that you can verify alignment.

   ```python
   with open(results_path / "retrieval_embeddings.pkl", "rb") as file:
       embedding_cache = pickle.load(file)

   assert embedding_cache["model_id"] == MODEL_ID
   assert embedding_cache["text_ids"] == df["text_id"].tolist()
   embeddings = embedding_cache["corpus_embeddings"]
   query_embeddings = embedding_cache["query_embeddings"]
   ```

2. **Continue directly with completed rankings.** Use this when you cannot load the
   model or when time is short.

   ```python
   with open(results_path / "retrieval_rankings.pkl", "rb") as file:
       ranking_cache = pickle.load(file)

   assert ranking_cache["model_id"] == MODEL_ID
   assert ranking_cache["text_ids"] == df["text_id"].tolist()
   ranking_1 = ranking_cache["ranking_1"].copy()
   ranking_2 = ranking_cache["ranking_2"].copy()
   ```

:::

## Rank the corpus for Query 1


In [ ]:
QUERY_1 = "People cannot afford a place to live."
TOP_K = 25

Retrieval has three steps: encode the query with the same model, compare its
embedding with every corpus embedding, and sort the documents by similarity.

::: {.callout-warning title="Similarity is a ranking signal"}

A high cosine similarity means that the model places a sentence close to the query.
It does not establish that the sentence satisfies our substantive definition of
housing affordability. There is no universal score above which a document becomes
relevant.

:::

In [ ]:
def rank_from_embedding(query_embedding, corpus, corpus_embeddings):
    """Return every corpus row ranked by cosine similarity to a query."""
    scores = corpus_embeddings @ query_embedding
    order = np.argsort(-scores)

    ranked = corpus.iloc[order][["text_id", "text"]].copy()
    ranked.insert(0, "rank", np.arange(1, len(ranked) + 1))
    ranked["similarity"] = scores[order]
    return ranked.reset_index(drop=True)

In [ ]:
# TODO: Embed Query 1 with the same normalization used for the corpus.
query_1_embedding = embedding_model.encode(
    [QUERY_1],
    normalize_embeddings=True,
)[0]

# If you loaded the embedding cache instead, use:
# query_1_embedding = query_embeddings["query_1"]

In [ ]:
# TODO: Rank the corpus and inspect the first ten results.
ranking_1 = rank_from_embedding(query_1_embedding, df, embeddings)
ranking_1.head(10)

## Exercise: judge Query 1's top 25

Work in pairs. Before assigning labels, write one or two sentences that define what
counts as relevant to **housing affordability**. Then classify every result as
`relevant`, `borderline`, or `irrelevant`.

Consider whether the sentence must explicitly mention price, rent, affordability,
or access to housing. Decide how to treat broader housing policy, homelessness,
housebuilding, planning, and statements that need surrounding context.

In [ ]:
relevance_rule = """
TODO: Write your relevance rule here.
""".strip()

judgment_table = ranking_1.head(TOP_K).copy()

# TODO: Replace the empty strings with "relevant", "borderline", or "irrelevant".
judgments = [""] * TOP_K
judgment_table["relevance_judgment"] = judgments

judgment_table

If all rows have been coded, this cell calculates the share judged relevant among
the first 25 results. It describes precision among these top results under your
relevance rule; it does not tell us how many relevant sentences were missed.

In [ ]:
allowed_judgments = {"relevant", "borderline", "irrelevant"}
observed_judgments = set(judgment_table["relevance_judgment"])

if observed_judgments.issubset(allowed_judgments) and "" not in observed_judgments:
    precision_at_25 = judgment_table["relevance_judgment"].eq("relevant").mean()
    print(f"Human-judged precision@25: {precision_at_25:.2f}")
else:
    print("Complete the relevance_judgment column before calculating precision@25.")

## Assess sensitivity to query wording

In [ ]:
QUERY_2 = "Housing costs place suitable homes beyond the reach of many households."

Query 2 expresses a similar concept with different words. To focus on interpretation
rather than another round of model inference, load its prepared ranking. If you
already loaded both cached rankings, this cell simply reuses `ranking_2`.

In [ ]:
if "ranking_2" not in globals():
    with open(results_path / "retrieval_rankings.pkl", "rb") as file:
        ranking_cache = pickle.load(file)
    assert ranking_cache["text_ids"] == df["text_id"].tolist()
    ranking_2 = ranking_cache["ranking_2"].copy()

In [ ]:
top_1 = ranking_1.head(TOP_K).rename(columns={
    "rank": "rank_query_1",
    "similarity": "similarity_query_1",
})
top_2 = ranking_2.head(TOP_K).rename(columns={
    "rank": "rank_query_2",
    "similarity": "similarity_query_2",
})

rank_comparison = top_1.merge(
    top_2,
    on=["text_id", "text"],
    how="outer",
    validate="one_to_one",
)
rank_comparison["status"] = np.select(
    [
        rank_comparison["rank_query_1"].notna()
        & rank_comparison["rank_query_2"].notna(),
        rank_comparison["rank_query_1"].notna(),
    ],
    ["retained", "dropped"],
    default="new",
)
rank_comparison["rank_change_query_2_minus_1"] = (
    rank_comparison["rank_query_2"] - rank_comparison["rank_query_1"]
)
rank_comparison = rank_comparison.sort_values(
    ["status", "rank_query_1", "rank_query_2"], na_position="last"
).reset_index(drop=True)

rank_comparison

In [ ]:
retained = rank_comparison.query("status == 'retained'")
overlap_at_25 = len(retained) / TOP_K
rank_correlation = retained["rank_query_1"].corr(
    retained["rank_query_2"], method="spearman"
)

pd.Series({
    "overlap@25": overlap_at_25,
    "Spearman correlation among retained results": rank_correlation,
})

Discuss:

- Which results remain highly ranked under both formulations?
- What distinguishes sentences that enter or leave the top 25?
- Does one query better match your relevance rule?
- What does this reveal about using a single query as a measurement instrument?

## Compare with the CAP topic labels

CAP topic 14 covers Community Development, Planning and Housing Issues. It is
broader than housing affordability, so it is an external benchmark rather than
unquestionable ground truth for this retrieval task.

In [ ]:
cap_lookup = df[["text_id", "cap_topic"]]
revealed_top25 = judgment_table.merge(
    cap_lookup, on="text_id", how="left", validate="one_to_one"
)
revealed_top25["CAP topic 14"] = revealed_top25["cap_topic"].eq(14)
revealed_top25

In [ ]:
topic14_top25 = revealed_top25["CAP topic 14"].mean()
topic14_corpus = df["cap_topic"].eq(14).mean()

pd.Series({
    "CAP topic 14 share in top 25": topic14_top25,
    "CAP topic 14 prevalence in corpus": topic14_corpus,
    "enrichment ratio": topic14_top25 / topic14_corpus,
})

In [ ]:
plot_data = revealed_top25.sort_values("rank", ascending=False)
colors = plot_data["CAP topic 14"].map({True: "#D55E00", False: "#999999"})

plt.figure(figsize=(8, 8))
plt.barh(plot_data["rank"].astype(str), plot_data["similarity"], color=colors)
plt.xlabel("Cosine similarity to Query 1")
plt.ylabel("Rank")
plt.title("Top-25 retrieval scores\norange = CAP topic 14")
plt.show()

The plot asks whether the ranking contains a signal related to the CAP housing
topic. Topic membership does not replace the narrower human judgment of relevance
to housing affordability.

## Optional appendix: a retrieval-optimized model

`all-MiniLM-L6-v2` is a compact general-purpose sentence-embedding model. The
larger `mixedbread-ai/mxbai-embed-large-v1` model is trained with retrieval in mind,
but takes longer to download and run.

In [ ]:
#| eval: false
ADVANCED_MODEL_ID = "mixedbread-ai/mxbai-embed-large-v1"
advanced_model = SentenceTransformer(ADVANCED_MODEL_ID)

advanced_embeddings = advanced_model.encode(
    df["text"].tolist(),
    batch_size=16,
    normalize_embeddings=True,
    show_progress_bar=True,
)
advanced_query = advanced_model.encode(
    [QUERY_1],
    prompt_name="query",
    normalize_embeddings=True,
)[0]

advanced_ranking = rank_from_embedding(advanced_query, df, advanced_embeddings)
advanced_ranking.head(25)

If you run the appendix, compare overlap and substantive relevance rather than raw
scores: similarity scales are not directly comparable across embedding models.